In [ ]:
import pandas as pd
from services import Hotmart
from aloud_database.aloud_database import Database
from datetime import datetime, timezone
from csv_readers.readers import tmb_csv_reader

In [ ]:
hotmart = Hotmart()
db = Database()
df_sales_to_check = pd.read_csv('data/sales_to_ajust.csv')b

In [ ]:
df_hotmart = df_sales_to_check[df_sales_to_check['platform'] == 'hotmart']

hotmart.get_sales_history(transaction=df_hotmart.iloc(0)[0].get('transaction_code'))

In [ ]:
import time

def timestamp_para_isostring(timestamp_ms):
    """Converte timestamp em milissegundos para string ISO 8601."""
    dt = datetime.fromtimestamp(timestamp_ms / 1000, tz=timezone.utc)
    return dt.isoformat()


def correct_data(conversion_id: str, new_date: str):
    query = f"""
        UPDATE lead_tracking_prod.conversions
        SET created_at = '{new_date}'::timestamptz
        WHERE id = '{conversion_id}'::uuid
    """

    result = db.execute_update(query=query)
    return result

# Loop melhorado com logs e intervalo de 0,5s entre requisições
for idx, row in df_hotmart.iterrows():
    try:
        print(f"Processando linha {idx} - transaction_code: {row['transaction_code']}")
        historico = hotmart.get_sales_history(transaction=row['transaction_code'])
        if not historico or not historico[0].get('purchase'):
            print(f"[ERRO] Não foi possível obter dados para transaction_code: {row['transaction_code']}")
            continue

        timestamp_ms = historico[0].get('purchase').get('approved_date')
        isostring = timestamp_para_isostring(timestamp_ms)
        print(f"Data aprovada encontrada: {isostring}")

        resultado = correct_data(conversion_id=row.get('id'), new_date=isostring)
        print(f"Atualização realizada para id {row.get('id')}: {resultado}")

        df_hotmart.at[idx, 'done'] = True
    except Exception as e:
        print(f"[ERRO] Falha ao processar linha {idx} (transaction_code: {row['transaction_code']}): {e}")
    time.sleep(0.5)

In [ ]:
df_sales_tmb = tmb_csv_reader()

df_tmb = df_sales_to_check[df_sales_to_check['platform'] == 'tmb']

In [ ]:
for idx, row in df_tmb.iterrows():
    try:
        transaction_code = row.get('transaction_code')
        print(f"Processando linha {idx} - transaction_code: {transaction_code}")

        try:
            transaction_code_int = int(transaction_code)
        except (ValueError, TypeError):
            print(f"[ERRO] transaction_code '{transaction_code}' não é um inteiro válido.")
            continue

        historico = df_sales_tmb[df_sales_tmb['Pedido'] == transaction_code_int]

        if historico.empty:
            print(f"[ERRO] Nenhuma venda encontrada para transaction_code: {transaction_code}")
            continue

        # Pega o primeiro valor correspondente
        criado_em = historico.iloc[0]['Criado Em']
        if pd.isnull(criado_em):
            print(f"[ERRO] Data 'Criado Em' ausente para transaction_code: {transaction_code}")
            continue

        isostring = criado_em.isoformat()
        print(f"Data aprovada encontrada: {isostring}")

        resultado = correct_data(conversion_id=row.get('id'), new_date=isostring)
        print(f"Atualização realizada para id {row.get('id')}: {resultado}")

        df_tmb.at[idx, 'done'] = True
    except Exception as e:
        print(f"[ERRO] Falha ao processar linha {idx} (transaction_code: {row.get('transaction_code')}): {e}")
    time.sleep(0.5)
